<a href="https://colab.research.google.com/github/Khalimovgeek/task_foundational/blob/master/imdb_review_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

In [ ]:
!pip install pyyaml h5py

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import layers
from tensorflow.keras import models
import os

In [ ]:
max_len = 150
max_features = 10000

def create_model():
  model = models.Sequential(
      [
          layers.Input(shape=(max_len,)),
          layers.Embedding(input_dim=max_features, output_dim=128),
          layers.LSTM(64),
          layers.Dense(64, activation='relu'),
          layers.Dense(1, activation='sigmoid')
      ]
  )

  model.compile(
      optimizer='adam',
      loss='binary_crossentropy',
      metrics=['accuracy']
  )

  return model



In [ ]:
(x_train,y_train), (x_test,y_test) = tf.keras.datasets.imdb.load_data(num_words=max_features)
x_train = pad_sequences(
    x_train, maxlen=max_len
)
x_test = pad_sequences(
    x_test, maxlen=max_len
)
split_index = int(len(x_train) * 0.8)
x_train, x_val = x_train[:split_index], x_train[split_index:]
y_train, y_val = y_train[:split_index], y_train[split_index:]



17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
checkpt_dir = "imdb_checkpoints"
os.makedirs(checkpt_dir, exist_ok=True)
checkpt_path = os.path.join(checkpt_dir, "cp-{epoch:04d}.weights.h5")
cp_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpt_path,
    verbose=1,
    save_weights_only=True,
    save_freq='epoch'
)

In [ ]:
model = create_model()
model.fit(
    x_train, y_train,
    batch_size=32,
    epochs=10,
    validation_data=(x_val, y_val),
    callbacks=[cp_callback]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7256 - loss: 0.5047
Epoch 1: saving model to imdb_checkpoints/cp-0001.weights.h5

Epoch 1: finished saving model to imdb_checkpoints/cp-0001.weights.h5
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 11ms/step - accuracy: 0.8085 - loss: 0.4107 - val_accuracy: 0.8536 - val_loss: 0.3409
Epoch 2/10
621/625 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9056 - loss: 0.2419
Epoch 2: saving model to imdb_checkpoints/cp-0002.weights.h5

Epoch 2: finished saving model to imdb_checkpoints/cp-0002.weights.h5
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9048 - loss: 0.2414 - val_accuracy: 0.8566 - val_loss: 0.3536
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9452 - loss: 0.1501
Epoch 3: saving model to imdb_checkpoints/cp-0003.weights.h5

Epoch 3: finished saving model to imdb_checkpoints/cp-0003.weights.h5
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.9374 - loss: 0.1661 - val_accuracy: 0.8380 - val_lo

In [ ]:
import glob
new_model = create_model()
# Search for the latest .weights.h5 file in the directory
files = glob.glob(os.path.join(checkpt_dir, "*.weights.h5"))

if files:
    latest = max(files, key=os.path.getmtime)
    print(f"\nLoading weights from: {latest}")
    new_model.load_weights(latest)

    # Evaluate the loaded model
    loss, acc = new_model.evaluate(x_test, y_test, verbose=2)
    print(f"\nRestored model - Loss: {loss:.4f}, Accuracy: {acc*100:.2f}%")
else:
    print("No checkpoint files found to load.")


Loading weights from: imdb_checkpoints/cp-0010.weights.h5


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


782/782 - 5s - 6ms/step - accuracy: 0.8450 - loss: 0.8098

Restored model - Loss: 0.8098, Accuracy: 84.50%


In [ ]:
word_index = tf.keras.datasets.imdb.get_word_index()

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def preprocess_text(text, max_len = 50):
  words  = text.lower().split()

  sequence = []
  for word in words:
    value = word_index.get(word)
    if value is not None and (value+3) <10000:
      sequence.append(value + 3)
    else:
      sequence.append(2)

  sequence = pad_sequences([sequence], maxlen=max_len)
  return sequence

In [ ]:
def predict_sentiment(model, review_text):
    # Preprocess the raw text input
    processed_input = preprocess_text(review_text)

    # Generate prediction (returns a probability score between 0.0 and 1.0)
    prediction = model.predict(processed_input, verbose=0)[0][0]

    # Classify as Positive (>= 0.5) or Negative (< 0.5)
    sentiment = "Positive" if prediction >= 0.5 else "Negative"

    print(f"Review: '{review_text}'")
    print(f"Confidence Score: {prediction:.4f}")
    print(f"Predicted Sentiment: {sentiment}\n")

In [ ]:
# Test examples
test_positive = "This movie was absolutely hateful the acting was great and I loved the story"
test_negative = "The plot made no sense and the acting was terrible I hated this film"

# Run predictions
predict_sentiment(new_model, test_positive)
predict_sentiment(new_model, test_negative)

Review: 'This movie was absolutely hateful the acting was great and I loved the story'
Confidence Score: 0.9984
Predicted Sentiment: Positive

Review: 'The plot made no sense and the acting was terrible I hated this film'
Confidence Score: 0.0083
Predicted Sentiment: Negative

